# Small Language Model: Next-Token Prediction from Scratch

In the previous notebook, we learned how Transformer blocks process token sequences.

In this notebook, we will build a small language model that learns a simple but important task:

given previous tokens, predict the next token.

This is the same basic training objective behind GPT-style models, but shown at a much smaller scale.

## Introduction

A language model estimates the probability of the next token given previous tokens. Autoregressive models generate text one token at a time: predict one token, append it to the context, then predict again.

GPT-style models are trained with next-token prediction at massive scale. This notebook keeps the model small so the core idea is easy to inspect and run on a CPU.

In [ ]:
import os
import random
import re
import string
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
except ImportError as exc:
    raise ImportError(
        "TensorFlow is required for this notebook. Install it with: pip install tensorflow"
    ) from exc

try:
    from IPython.display import display
except ImportError:
    display = print

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.threading.set_intra_op_parallelism_threads(2)
tf.config.threading.set_inter_op_parallelism_threads(2)

plt.style.use("default")
plt.rcParams.update({
    "figure.figsize": (10, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

print("TensorFlow version:", tf.__version__)

## What is Next-Token Prediction?

Example:

Input:

`The model learns from`

Target:

`data`

The model receives a context window and learns to predict what comes next. During training, each example contains previous tokens as input and the next token as the target.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis("off")

context_tokens = ["The", "model", "learns", "from"]
target_token = "data"

for idx, token in enumerate(context_tokens):
    ax.text(
        idx * 1.25,
        0.6,
        token,
        ha="center",
        va="center",
        bbox=dict(boxstyle="round,pad=0.35", facecolor="#eef5fb", edgecolor="#3b6ea8"),
        fontsize=11,
    )
    if idx < len(context_tokens) - 1:
        ax.annotate("", xy=(idx * 1.25 + 0.52, 0.6), xytext=(idx * 1.25 + 0.72, 0.6), arrowprops=dict(arrowstyle="->"))

ax.annotate("predict", xy=(5.25, 0.6), xytext=(4.25, 0.6), arrowprops=dict(arrowstyle="->", lw=1.5))
ax.text(5.9, 0.6, target_token, ha="center", va="center", bbox=dict(boxstyle="round,pad=0.35", facecolor="#f2f7ef", edgecolor="#4b8f4b"), fontsize=11)
ax.text(1.9, 0.15, "context", ha="center", fontsize=10)
ax.text(5.9, 0.15, "target next token", ha="center", fontsize=10)
ax.set_xlim(-0.6, 6.8)
ax.set_ylim(0, 1)
ax.set_title("Next-Token Prediction")
plt.show()

## Tiny Training Corpus

The corpus below is intentionally small and realistic. It includes short sentences from machine learning, software engineering, cybersecurity, product support, data science, and documentation.

A real language model would need far more data, but a tiny corpus is enough to demonstrate the mechanics of next-token prediction.

In [ ]:
corpus = [
    "The model learns from data",
    "The model predicts the next token",
    "The model updates weights during training",
    "Machine learning systems improve with clean data",
    "Neural networks learn useful patterns from examples",
    "Embeddings convert tokens into dense vectors",
    "Attention helps models focus on relevant context",
    "Training loss decreases when predictions improve",
    "Validation accuracy measures generalization on held out data",
    "A small model can demonstrate language modeling",
    "Software engineers write tests before deployment",
    "Code reviews improve reliability and maintainability",
    "APIs should return clear error messages",
    "Documentation explains setup steps and expected behavior",
    "Continuous integration runs tests after each commit",
    "Version control tracks changes across the project",
    "Data pipelines clean transform and validate records",
    "Feature engineering can improve classical models",
    "Dashboards summarize metrics for product teams",
    "Analysts inspect distributions before training models",
    "Customer support teams resolve tickets quickly",
    "Support agents need context from previous messages",
    "A helpful response should answer the user clearly",
    "Product feedback reveals common user pain points",
    "Release notes describe fixes improvements and known issues",
    "Security teams monitor suspicious login attempts",
    "Cybersecurity alerts require fast investigation",
    "Authentication protects accounts from unauthorized access",
    "Encryption keeps sensitive data safer in transit",
    "Audit logs help teams investigate incidents",
    "The database stores events for later analysis",
    "The pipeline validates schema before loading data",
    "The classifier predicts labels from input features",
    "The tokenizer converts text into token ids",
    "The language model generates one token at a time",
    "The context window limits how much text the model sees",
    "Temperature controls randomness during generation",
    "Low temperature makes predictions more deterministic",
    "High temperature creates more diverse outputs",
    "Autoregressive generation repeats prediction and appending",
    "The assistant summarizes logs for engineers",
    "The monitoring service reports latency spikes",
    "Security teams review audit logs daily",
    "Customer support writes clear troubleshooting steps",
    "Data scientists compare models using validation metrics",
]

corpus_df = pd.DataFrame({"sentence": corpus})
corpus_df["word_count"] = corpus_df["sentence"].str.split().str.len()
display(corpus_df.head(12))
print("Number of sentences:", len(corpus_df))

## Tokenization and Vocabulary

This notebook uses a simple educational tokenizer:

- lowercase text
- remove punctuation
- split on whitespace
- build a vocabulary from the corpus
- map tokens to integer IDs

Production LLMs usually use subword tokenizers. Here, word-level tokenization keeps the training objective easy to see.

In [ ]:
def preprocess_text(text):
    """Lowercase text, remove punctuation, and normalize whitespace."""
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    """Tokenize text with simple whitespace splitting."""
    cleaned = preprocess_text(text)
    return cleaned.split() if cleaned else []


def build_vocabulary(sentences, special_tokens=None):
    """Build token-to-id and id-to-token mappings."""
    if special_tokens is None:
        special_tokens = ["<PAD>", "<UNK>"]

    counts = Counter()
    for sentence in sentences:
        counts.update(tokenize(sentence))

    vocabulary = list(special_tokens) + sorted(counts.keys())
    token_to_id = {token: idx for idx, token in enumerate(vocabulary)}
    id_to_token = {idx: token for token, idx in token_to_id.items()}
    return token_to_id, id_to_token, counts


token_to_id, id_to_token, token_counts = build_vocabulary(corpus)
vocab_size = len(token_to_id)

print("Vocabulary size:", vocab_size)
print("Most common tokens:", token_counts.most_common(10))

In [ ]:
vocab_df = pd.DataFrame({
    "token": list(token_to_id.keys()),
    "token_id": list(token_to_id.values()),
    "frequency": [token_counts.get(token, 0) for token in token_to_id.keys()],
})

display(vocab_df.head(25))

## Creating Training Sequences

A sliding window turns each sentence into multiple training examples.

For tokens:

`["the", "model", "learns", "from", "data"]`

with `context_size = 4`:

Input: `["the", "model", "learns", "from"]`

Target: `"data"`

In [ ]:
def encode_tokens(tokens, token_to_id):
    """Convert tokens into token IDs, using <UNK> for unknown tokens."""
    unk_id = token_to_id["<UNK>"]
    return [token_to_id.get(token, unk_id) for token in tokens]


def create_training_examples(sentences, token_to_id, context_size=4):
    """Create sliding-window next-token prediction examples."""
    examples = []
    for sentence in sentences:
        tokens = tokenize(sentence)
        if len(tokens) <= context_size:
            continue

        token_ids = encode_tokens(tokens, token_to_id)
        for start in range(0, len(tokens) - context_size):
            input_tokens = tokens[start:start + context_size]
            target_token = tokens[start + context_size]
            input_ids = token_ids[start:start + context_size]
            target_id = token_ids[start + context_size]
            examples.append({
                "input_tokens": input_tokens,
                "target_token": target_token,
                "input_ids": input_ids,
                "target_id": target_id,
                "source_sentence": sentence,
            })
    return examples


context_size = 4
training_examples = create_training_examples(corpus, token_to_id, context_size=context_size)
examples_df = pd.DataFrame(training_examples)
display(examples_df.head(12))
print("Training examples:", len(examples_df))

In [ ]:
example_tokens = ["the", "model", "learns", "from", "data"]
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis("off")

for idx, token in enumerate(example_tokens):
    color = "#eef5fb" if idx < context_size else "#f2f7ef"
    edge = "#3b6ea8" if idx < context_size else "#4b8f4b"
    ax.text(idx, 0.6, token, ha="center", va="center", bbox=dict(boxstyle="round,pad=0.35", facecolor=color, edgecolor=edge))

ax.plot([0, context_size - 1], [0.25, 0.25], color="#3b6ea8", lw=3)
ax.text(1.5, 0.08, "input window", ha="center")
ax.text(context_size, 0.08, "target", ha="center")
ax.set_xlim(-0.6, len(example_tokens) - 0.4)
ax.set_ylim(0, 1)
ax.set_title("Sliding Window Sequence Creation")
plt.show()

## Dataset Preparation

The model will receive input token IDs with shape `(examples, context_size)` and target IDs with shape `(examples,)`.

The output layer will have one probability for every token in the vocabulary.

In [ ]:
X = np.array(examples_df["input_ids"].tolist(), dtype=np.int32)
y = np.array(examples_df["target_id"].tolist(), dtype=np.int32)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Vocabulary size:", vocab_size)
print("Context size:", context_size)

## Building a Small Neural Language Model

This is intentionally not a Transformer yet. The goal is to understand the next-token objective.

Architecture:

- Embedding: converts token IDs into dense vectors
- Flatten: combines context token embeddings into one vector
- Dense hidden layer: learns simple patterns in the context
- Dense softmax output: predicts a probability distribution over the vocabulary

In [ ]:
embedding_dim = 24
hidden_units = 32

model = models.Sequential([
    layers.Input(shape=(context_size,), dtype="int32"),
    layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, name="token_embedding"),
    layers.Flatten(name="flatten_context"),
    layers.Dense(hidden_units, activation="relu", name="hidden_layer"),
    layers.Dense(vocab_size, activation="softmax", name="next_token_distribution"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
    run_eagerly=True,
)

## Model Summary

The output layer has `vocabulary_size` units. Each unit represents the model's predicted probability for one possible next token.

In [ ]:
model.summary()

## Training

The model trains on sliding-window examples. Because the corpus is tiny, this is a demonstration of the objective rather than a useful real-world model.

In [ ]:
num_epochs = 40
validation_fraction = 0.2

rng = np.random.default_rng(SEED)
indices = rng.permutation(len(X))
split_index = int(len(indices) * (1 - validation_fraction))
train_indices = indices[:split_index]
val_indices = indices[split_index:]

X_train = tf.constant(X[train_indices], dtype=tf.int32)
y_train = tf.constant(y[train_indices], dtype=tf.int32)
X_val = tf.constant(X[val_indices], dtype=tf.int32)
y_val = tf.constant(y[val_indices], dtype=tf.int32)

optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

for epoch in range(num_epochs):
    with tf.GradientTape() as tape:
        train_probabilities = model(X_train, training=True)
        train_loss = loss_fn(y_train, train_probabilities)

    gradients = tape.gradient(train_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    val_probabilities = model(X_val, training=False)
    val_loss = loss_fn(y_val, val_probabilities)

    train_accuracy = np.mean(
        tf.keras.metrics.sparse_categorical_accuracy(y_train, train_probabilities).numpy()
    )
    val_accuracy = np.mean(
        tf.keras.metrics.sparse_categorical_accuracy(y_val, val_probabilities).numpy()
    )

    history["loss"].append(float(train_loss.numpy()))
    history["accuracy"].append(float(train_accuracy))
    history["val_loss"].append(float(val_loss.numpy()))
    history["val_accuracy"].append(float(val_accuracy))

final_loss = history["loss"][-1]
final_accuracy = history["accuracy"][-1]
final_val_loss = history["val_loss"][-1]
final_val_accuracy = history["val_accuracy"][-1]

print(f"Final training loss: {final_loss:.4f}")
print(f"Final training accuracy: {final_accuracy:.4f}")
print(f"Final validation loss: {final_val_loss:.4f}")
print(f"Final validation accuracy: {final_val_accuracy:.4f}")


## Training Curves

Loss measures prediction error. Accuracy measures how often the top predicted token matches the target token.

With a tiny corpus, training accuracy can rise quickly while validation accuracy may remain unstable.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, len(history["loss"]) + 1)

axes[0].plot(epochs, history["loss"], label="train loss", color="#3b6ea8")
axes[0].plot(epochs, history["val_loss"], label="validation loss", color="#d17a22")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(epochs, history["accuracy"], label="train accuracy", color="#4b8f4b")
axes[1].plot(epochs, history["val_accuracy"], label="validation accuracy", color="#9b4d96")
axes[1].set_title("Training Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

## Predicting the Next Token

The function below preprocesses a context, encodes the most recent `context_size` tokens, handles unknown words safely, and returns the top candidate next tokens with probabilities.

In [ ]:
def prepare_context(context_text, token_to_id, context_size):
    """Convert context text into a fixed-size model input."""
    tokens = tokenize(context_text)
    ids = encode_tokens(tokens, token_to_id)

    pad_id = token_to_id["<PAD>"]
    if len(ids) < context_size:
        ids = [pad_id] * (context_size - len(ids)) + ids
    else:
        ids = ids[-context_size:]

    return np.array([ids], dtype=np.int32), tokens


def predict_next_token(context_text, top_k=5):
    """Predict top-k next tokens for a text context."""
    model_input, tokens = prepare_context(context_text, token_to_id, context_size)
    probabilities = model(model_input, training=False).numpy()[0]

    excluded_ids = {token_to_id["<PAD>"], token_to_id["<UNK>"]}
    candidate_ids = [idx for idx in np.argsort(probabilities)[::-1] if idx not in excluded_ids]
    top_ids = candidate_ids[:top_k]

    return pd.DataFrame({
        "context_tokens": [tokens] * len(top_ids),
        "candidate_token": [id_to_token[idx] for idx in top_ids],
        "probability": [probabilities[idx] for idx in top_ids],
    })

prediction_df = predict_next_token("the model learns from", top_k=5)
display(prediction_df)

## Visualizing Token Probabilities

A softmax layer returns a full probability distribution. The chart below shows the top candidates for one context.

In [ ]:
top_predictions = predict_next_token("security teams", top_k=8)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(top_predictions["candidate_token"], top_predictions["probability"], color="#3b6ea8")
ax.set_title("Top-K Next Token Probabilities")
ax.set_xlabel("Candidate token")
ax.set_ylabel("Probability")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

display(top_predictions)

## Autoregressive Text Generation

Autoregressive generation repeats the same step:

predict one token → append it → use the updated context → predict again

In [ ]:
def sample_from_probabilities(probabilities, temperature=1.0):
    """Sample a token ID from a probability distribution with temperature."""
    if temperature <= 0:
        raise ValueError("temperature must be positive")

    probabilities = np.asarray(probabilities).astype("float64")
    logits = np.log(np.maximum(probabilities, 1e-12)) / temperature
    adjusted = np.exp(logits - np.max(logits))
    adjusted = adjusted / adjusted.sum()
    return np.random.choice(len(adjusted), p=adjusted)


def generate_text(seed_text, num_tokens=10, temperature=1.0):
    """Generate text by repeatedly sampling next tokens."""
    generated_tokens = tokenize(seed_text)

    for _ in range(num_tokens):
        context = " ".join(generated_tokens)
        model_input, _ = prepare_context(context, token_to_id, context_size)
        probabilities = model(model_input, training=False).numpy()[0]

        probabilities[token_to_id["<PAD>"]] = 0
        probabilities[token_to_id["<UNK>"]] = 0
        probabilities = probabilities / probabilities.sum()

        next_id = sample_from_probabilities(probabilities, temperature=temperature)
        generated_tokens.append(id_to_token[next_id])

    return " ".join(generated_tokens)

for seed in ["the model", "security teams", "customer support"]:
    print(seed, "->", generate_text(seed, num_tokens=8, temperature=1.0))

In [ ]:
flow_steps = ["Seed text", "Predict next token", "Append token", "Update context", "Repeat"]
fig, ax = plt.subplots(figsize=(11, 2.6))
ax.axis("off")
ax.set_xlim(0, len(flow_steps))
ax.set_ylim(0, 1)

for idx, step in enumerate(flow_steps):
    ax.text(
        idx + 0.42,
        0.55,
        step,
        ha="center",
        va="center",
        bbox=dict(boxstyle="round,pad=0.35", facecolor="#f5f8fb", edgecolor="#3b6ea8"),
    )
    if idx < len(flow_steps) - 1:
        ax.annotate("", xy=(idx + 0.86, 0.55), xytext=(idx + 0.68, 0.55), arrowprops=dict(arrowstyle="->"))

ax.set_title("Autoregressive Generation Flow")
plt.show()

## Temperature Intuition

Temperature changes how the probability distribution is sampled.

- Low temperature makes output more deterministic.
- Medium temperature follows the learned distribution more naturally.
- High temperature increases randomness and can become incoherent.

In [ ]:
temperature_rows = []
for temperature in [0.5, 1.0, 1.5]:
    np.random.seed(SEED)
    generated = generate_text("the model", num_tokens=10, temperature=temperature)
    temperature_rows.append({
        "temperature": temperature,
        "generated_text": generated,
    })

temperature_df = pd.DataFrame(temperature_rows)
display(temperature_df)

## Mini Project: Tiny Autocomplete Demo

The `autocomplete` function returns top predicted next tokens, their probabilities, and a generated continuation for the prompt.

In [ ]:
def autocomplete(prompt, top_k=5, continuation_tokens=8, temperature=0.8):
    """Return autocomplete candidates and a generated continuation."""
    candidates = predict_next_token(prompt, top_k=top_k)
    continuation = generate_text(
        prompt,
        num_tokens=continuation_tokens,
        temperature=temperature,
    )
    return {
        "prompt": prompt,
        "top_predictions": candidates,
        "generated_continuation": continuation,
    }

prompts = ["the model", "security teams", "customer support", "data pipelines"]
autocomplete_rows = []

for prompt in prompts:
    result = autocomplete(prompt)
    top_tokens = result["top_predictions"]["candidate_token"].tolist()
    top_probs = result["top_predictions"]["probability"].round(3).tolist()
    autocomplete_rows.append({
        "prompt": prompt,
        "top_tokens": top_tokens,
        "top_probabilities": top_probs,
        "generated_continuation": result["generated_continuation"],
    })

autocomplete_df = pd.DataFrame(autocomplete_rows)
display(autocomplete_df)

## Limitations of Small Language Models

This model is useful for learning, but it is not comparable to a real LLM.

Limitations include:

- tiny corpus
- limited vocabulary
- weak generalization
- no real-world knowledge
- no long-context understanding
- possible memorization of training patterns
- no Transformer decoder blocks

## How This Connects to Real LLMs

Real LLMs use massive datasets, subword tokenizers, Transformer decoder blocks, billions of parameters, next-token prediction, instruction tuning, and alignment.

The scale is very different, but the core objective remains similar: learn to predict the next token from context.

## Final Summary

Key takeaways:

- language modeling predicts next tokens
- training data becomes input-target pairs
- softmax produces token probabilities
- autoregressive generation repeats prediction and appending
- temperature controls randomness during generation

## Next Notebook

➡️ Next: Temperature and Autoregressive Generation

The next notebook will focus more deeply on decoding strategies, temperature, sampling, and generation behavior.